In [1]:
!pip install datasets evaluate rouge-score bert-score tqdm pandas -q

In [2]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from peft import PeftModel
import pandas as pd
from tqdm import tqdm
from evaluate import load
from bert_score import score as bertscore

/common/home/projectgrps/CS425/CS425G9/jupyterlab-venv-pytorch-240/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### T5 + LoRA

In [3]:
# --- Paths ---
base_model_name = "./flan_t5_base_local"   # t5 base
lora_path = "./flan_t5_med_lora_v8"        # t5 + lora
test_path = "./medical_qa_test.csv"        # test dataset

# --- Load model and tokenizer ---
tokenizer = AutoTokenizer.from_pretrained(base_model_name)
base_model = AutoModelForSeq2SeqLM.from_pretrained(base_model_name)
model = PeftModel.from_pretrained(base_model, lora_path)
model = model.to("cuda")

# --- Load dataset ---
df = pd.read_csv(test_path)
rouge = load("rouge")

preds, refs, qns = [], [], []

print("Conducting inference for eval set")
for _, row in tqdm(df.iterrows(), total=len(df)):
    q = str(row["Question"]) if pd.notna(row["Question"]) else ""
    a = str(row["Answer"]) if pd.notna(row["Answer"]) else ""
    if not q.strip() or not a.strip():
        continue  # skip empty rows

    prompt = f"Question: {q} Answer:"
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    output = model.generate(**inputs, max_new_tokens=150)
    gen = tokenizer.decode(output[0], skip_special_tokens=True)

    # ensure generated text is string
    qns.append(str(q))
    preds.append(str(gen))
    refs.append(str(a))

# Safety check before computing ROUGE
if len(preds) == 0:
    raise ValueError("No predictions were generated — check your CSV formatting!")

df_out = pd.DataFrame({
    "question": qns,
    "reference": refs,
    "prediction": preds,
})
df_out.to_csv("v8_inference.csv", index=False)
print("saved to v8_inference.csv")

print("Computing ROUGE results")
result = rouge.compute(predictions=preds, references=refs)
print("ROUGE Results:")
print(f"  ROUGE-1   = {result['rouge1']:.5f}")
print(f"  ROUGE-2   = {result['rouge2']:.5f}")
print(f"  ROUGE-L   = {result['rougeL']:.5f}")
print(f"  ROUGE-Lsum= {result['rougeLsum']:.5f}")

Conducting inference for eval set


  0%|          | 0/22211 [00:00<?, ?it/s]

  0%|          | 1/22211 [00:01<8:35:28,  1.39s/it]

  0%|          | 2/22211 [00:02<7:08:48,  1.16s/it]

  0%|          | 3/22211 [00:03<6:42:29,  1.09s/it]

...

100%|█████████▉| 22210/22211 [6:06:28<00:00,  1.11it/s]

100%|██████████| 22211/22211 [6:06:30<00:00,  1.04it/s]

100%|██████████| 22211/22211 [6:06:30<00:00,  1.01it/s]

saved to v8_inference.csv
Computing ROUGE results


ROUGE Results:
  ROUGE-1   = 0.15901
  ROUGE-2   = 0.02991
  ROUGE-L   = 0.11650
  ROUGE-Lsum= 0.11648


In [4]:
# BERTScore for semantic evaluation

P, R, F1 = bertscore(preds, refs, lang="en", verbose=True)
p_mean, r_mean, f1_mean = P.mean().item(), R.mean().item(), F1.mean().item()

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


calculating scores...
computing bert embedding.


  0%|          | 0/672 [00:00<?, ?it/s]

  0%|          | 1/672 [00:00<03:34,  3.13it/s]

  0%|          | 2/672 [00:00<03:24,  3.28it/s]

...

 90%|████████▉ | 312/347 [00:02<00:00, 127.14it/s]

 94%|█████████▎| 325/347 [00:02<00:00, 127.12it/s]

 97%|█████████▋| 338/347 [00:02<00:00, 126.81it/s]

100%|██████████| 347/347 [00:02<00:00, 126.09it/s]

done in 112.51 seconds, 197.34 sentences/sec


In [5]:
print(f"  Average BERTScore F1 {F1.mean().item():.5f}")
print(f"  Precision    = {p_mean:.5f}\n")
print(f"  Recall    = {r_mean:.5f}\n")
print(f"  F1        = {f1_mean:.5f}\n\n")

  Average BERTScore F1 0.81719
  Precision    = 0.81548

  Recall    = 0.81931

  F1        = 0.81719




In [6]:
import torch, math

model.eval()
neg_log_likelihood_sum = 0.0
total_target_tokens = 0

for q, a in zip(qns, refs):
    prompt = f"Question: {q} Answer:"
    enc = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=256).to("cuda")
    lab = tokenizer(a, return_tensors="pt", truncation=True, max_length=256,
                    padding="longest").to("cuda")
    labels = lab["input_ids"]
    labels = labels.masked_fill(labels == tokenizer.pad_token_id, -100)

    with torch.no_grad():
        out = model(**enc, labels=labels)
    # out.loss is mean loss per non-ignored token
    n_tokens = (labels != -100).sum().item()
    if n_tokens == 0:
        continue  # skip this example
    neg_log_likelihood_sum += out.loss.item() * n_tokens
    total_target_tokens += n_tokens

if total_target_tokens == 0:
    raise ValueError("No valid target tokens to compute perplexity")

ppl = math.exp(neg_log_likelihood_sum / total_target_tokens)
print(f"Dataset Perplexity: {ppl:.6f}")

Dataset Perplexity: 18.284920


In [7]:
with open("rouge_results_v8.txt", "w") as f:
    f.write("=== ROUGE-v8 ===\n")
    f.write(f"  ROUGE-1   = {result['rouge1']:.5f}\n")
    f.write(f"  ROUGE-2   = {result['rouge2']:.5f}\n")
    f.write(f"  ROUGE-L   = {result['rougeL']:.5f}\n")
    f.write(f"  ROUGE-Lsum= {result['rougeLsum']:.5f}\n")
    f.write(f"  Average BERTScore F1 {F1.mean().item():.5f}")
    f.write(f"  Precision    = {p_mean:.5f}\n")
    f.write(f"  Recall    = {r_mean:.5f}\n")
    f.write(f"  F1        = {f1_mean:.5f}\n\n")
    f.write(f"  Dataset Perplexity: {ppl:.6f}")

### Base Model

In [8]:
# base_model_name = "./flan_t5_base_local"
# test_path = "./medical_qa_test.csv"

# # --- Load base model and tokenizer (no LoRA) ---
# tokenizer = AutoTokenizer.from_pretrained(base_model_name)
# model = AutoModelForSeq2SeqLM.from_pretrained(base_model_name).to("cuda")

# # --- Load dataset ---
# df = pd.read_csv(test_path, usecols=["Question", "Answer"])
# rouge = load("rouge")

# preds, refs = [], []

# for _, row in tqdm(df.iterrows(), total=len(df)):
#     q = str(row["Question"]) if pd.notna(row["Question"]) else ""
#     a = str(row["Answer"]) if pd.notna(row["Answer"]) else ""
#     if not q.strip() or not a.strip():
#         continue

#     prompt = f"Question: {q} Answer:"
#     inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
#     output = model.generate(**inputs, max_new_tokens=150)
#     gen = tokenizer.decode(output[0], skip_special_tokens=True)
#     preds.append(gen)
#     refs.append(a)

# # --- Compute ROUGE ---
# result_base = rouge.compute(predictions=preds, references=refs)
# print("Base Model ROUGE Results:")
# print(f"  ROUGE-1   = {result_base['rouge1']:.5f}")
# print(f"  ROUGE-2   = {result_base['rouge2']:.5f}")
# print(f"  ROUGE-L   = {result_base['rougeL']:.5f}")
# print(f"  ROUGE-Lsum= {result_base['rougeLsum']:.5f}")

In [9]:
# P, R, F1 = bertscore(preds, refs, lang="en", verbose=True)
# p_mean, r_mean, f1_mean = P.mean().item(), R.mean().item(), F1.mean().item()

In [10]:
# with open("rouge_results_base.txt", "w") as f:
#     f.write("=== ROUGE-base ===\n")
#     f.write(f"  ROUGE-1   = {result['rouge1']:.5f}\n")
#     f.write(f"  ROUGE-2   = {result['rouge2']:.5f}\n")
#     f.write(f"  ROUGE-L   = {result['rougeL']:.5f}\n")
#     f.write(f"  ROUGE-Lsum= {result['rougeLsum']:.5f}\n")
#     f.write(f"  Precision    = {p_mean:.5f}\n")
#     f.write(f"  Recall    = {r_mean:.5f}\n")
#     f.write(f"  F1        = {f1_mean:.5f}\n\n")